# 08 · Text-to-Image

> **Source notes:** `TextToImage.md`

Prompt engineering, img2img, and spatial control — from text to pictures.

This notebook:
- Implements **img2img** on our MNIST CondUNet (partial noise + denoise)
- Visualises how **strength** controls how much of the source image is preserved
- Demonstrates **negative conditioning** (the negative-prompt extension of CFG)
- Simulates a **ControlNet**-style structural injection (edge map as extra input)
- PixelSmith v5 bonus: real ControlNet via `diffusers` (commented, optional)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch", "torchvision", "matplotlib", "numpy", "-q"], check=True)
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
import torchvision.transforms as T
import numpy as np, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
print("Ready.")

## 1 · Rebuild Conditional U-Net + Train (from Ch.5/6)

In [ ]:
T_STEPS = 1000
betas     = torch.linspace(1e-4, 0.02, T_STEPS)
alphas    = 1 - betas
alpha_bar = torch.cumprod(alphas, 0)
sqrt_ab   = alpha_bar.sqrt()
sqrt_1mab = (1 - alpha_bar).sqrt()

NUM_CLASSES, NULL_CLASS = 10, 10

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half  = self.dim // 2
        freqs = torch.exp(-torch.arange(half, device=t.device) * (np.log(10000)/(half-1)))
        args  = t[:,None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1  = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(time_dim, out_ch)
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = F.silu(self.conv1(x))
        h = h + self.t_proj(t_emb)[:, :, None, None]
        return F.silu(self.conv2(h)) + self.skip(x)

class CondUNet(nn.Module):
    def __init__(self, base_ch=32, time_dim=64):
        super().__init__()
        self.class_embed = nn.Embedding(NUM_CLASSES + 1, time_dim)
        self.time_embed  = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim*4), nn.SiLU(),
            nn.Linear(time_dim*4, time_dim))
        C = base_ch
        self.enc1 = ResBlock(1, C,   time_dim)
        self.enc2 = ResBlock(C, C*2, time_dim)
        self.enc3 = ResBlock(C*2, C*4, time_dim)
        self.bot  = ResBlock(C*4, C*4, time_dim)
        self.dec3 = ResBlock(C*8, C*2, time_dim)
        self.dec2 = ResBlock(C*4, C,   time_dim)
        self.dec1 = ResBlock(C*2, C,   time_dim)
        self.out  = nn.Conv2d(C, 1, 1)
        self.down = nn.MaxPool2d(2)
        self.up   = nn.Upsample(scale_factor=2, mode="nearest")
    def forward(self, x, t, c):
        te = self.time_embed(t) + self.class_embed(c)
        s1 = self.enc1(x, te); s2 = self.enc2(self.down(s1), te)
        s3 = self.enc3(self.down(s2), te); b = self.bot(s3, te)
        d3 = self.dec3(torch.cat([self.up(b), s3], 1), te)
        d2 = self.dec2(torch.cat([self.up(d3), s2], 1), te)
        d1 = self.dec1(torch.cat([d2, s1], 1), te)
        return self.out(d1)

model  = CondUNet()
optim  = torch.optim.Adam(model.parameters(), lr=2e-4)
dataset = torchvision.datasets.MNIST(
    "/tmp/mnist", train=True, download=True,
    transform=T.Compose([T.ToTensor(), T.Lambda(lambda x: x*2-1)]))
loader = DataLoader(dataset, batch_size=256, shuffle=True)

model.train()
for epoch in range(20):
    for x0, c in loader:
        t = torch.randint(0, T_STEPS, (len(x0),))
        eps = torch.randn_like(x0)
        xt  = sqrt_ab[t].view(-1,1,1,1)*x0 + sqrt_1mab[t].view(-1,1,1,1)*eps
        mask = torch.rand(len(x0)) < 0.10
        c_in = c.clone(); c_in[mask] = NULL_CLASS
        loss = F.mse_loss(model(xt, t, c_in), eps)
        optim.zero_grad(); loss.backward(); optim.step()
    if (epoch+1) % 5 == 0: print(f"Epoch {epoch+1}/20  loss={loss.item():.4f}")
print("Training done.")

## 2 · img2img — Strength Sweep

Start from a real MNIST digit but ask the model to re-draw it as a different class.

In [ ]:
# Get a real digit '0' to use as source
for x_src, y_src in DataLoader(dataset, batch_size=64, shuffle=True):
    zeros = x_src[y_src == 0][:1]   # one real '0'
    if len(zeros): break

@torch.no_grad()
def img2img_ddim(model, x_src, target_class, strength=0.7, n_steps=30, w=3.0):
    """
    img2img with DDIM.
    strength: fraction of total timesteps to use (0=no change, 1=full T2I)
    """
    model.eval()
    N     = x_src.shape[0]
    cls   = torch.full((N,), target_class, dtype=torch.long)
    null  = torch.full((N,), NULL_CLASS,   dtype=torch.long)
    
    # Forward: add noise up to t_start
    t_start = int(strength * T_STEPS)
    eps_fwd = torch.randn_like(x_src)
    x_noisy = sqrt_ab[t_start]*x_src + sqrt_1mab[t_start]*eps_fwd
    
    # Reverse: denoise from t_start to 0
    x = x_noisy
    ts = torch.linspace(t_start - 1, 0, n_steps).long()
    for i in range(len(ts) - 1):
        t_cur  = ts[i]; t_next = ts[i+1]
        tb     = torch.full((N,), t_cur, dtype=torch.long)
        eps_c  = model(x, tb, cls); eps_u = model(x, tb, null)
        eps    = eps_u + w * (eps_c - eps_u)
        ab_c   = alpha_bar[t_cur]; ab_n = alpha_bar[t_next]
        x0_p   = ((x - sqrt_1mab[t_cur]*eps) / sqrt_ab[t_cur]).clamp(-1,1)
        x      = ab_n.sqrt()*x0_p + (1-ab_n).sqrt()*eps
    return x

strengths    = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
target_class = 8   # turn '0' into '8'

results = [img2img_ddim(model, zeros, target_class, s) for s in strengths]

fig, axes = plt.subplots(1, len(strengths)+1, figsize=(14, 2.5))
axes[0].imshow(zeros[0,0].numpy(), cmap="gray", vmin=-1, vmax=1)
axes[0].set_title("Source: '0'"); axes[0].axis("off")
for i, (s, img) in enumerate(zip(strengths, results)):
    axes[i+1].imshow(img[0,0].numpy(), cmap="gray", vmin=-1, vmax=1)
    axes[i+1].set_title(f"s={s:.1f}")
    axes[i+1].axis("off")

plt.suptitle("img2img: source='0', target='8', varying strength")
plt.tight_layout(); plt.show()
print("Low strength → original preserved. High strength → target concept dominates.")

## 3 · Negative Conditioning

Replace the null-token unconditional with a *negative class* baseline. This pushes the output away from an unwanted class.

In [ ]:
@torch.no_grad()
def cfg_negative_sample(model, target_class, negative_class, w=5.0, n=4, n_steps=40):
    """
    Extended CFG with explicit negative class (analogous to negative prompts).
    eps_hat = eps_neg + w * (eps_pos - eps_neg)
    """
    model.eval()
    pos  = torch.full((n,), target_class,  dtype=torch.long)
    neg  = torch.full((n,), negative_class, dtype=torch.long)
    x    = torch.randn(n, 1, 28, 28)
    ts   = torch.linspace(T_STEPS-1, 0, n_steps+1).long()
    for i in range(n_steps):
        t_cur  = ts[i]; t_next = ts[i+1]
        tb     = torch.full((n,), t_cur, dtype=torch.long)
        eps_p  = model(x, tb, pos)
        eps_n  = model(x, tb, neg)
        eps    = eps_n + w * (eps_p - eps_n)
        ab_c   = alpha_bar[t_cur]; ab_n = alpha_bar[t_next]
        x0_p   = ((x - sqrt_1mab[t_cur]*eps) / sqrt_ab[t_cur]).clamp(-1,1)
        x      = ab_n.sqrt()*x0_p + (1-ab_n).sqrt()*eps
    return x

# Generate '3' while using '8' as the negative (avoids loopy shapes)
target   = 3
neg_classes = [NULL_CLASS, 0, 6, 8, 9]  # 10=null=standard CFG

fig, axes = plt.subplots(len(neg_classes), 4, figsize=(7, len(neg_classes)*2.2))
for row, neg in enumerate(neg_classes):
    imgs = cfg_negative_sample(model, target, neg, w=5.0)
    neg_label = "null (standard)" if neg == NULL_CLASS else str(neg)
    for col in range(4):
        axes[row, col].imshow(imgs[col, 0].numpy(), cmap="gray", vmin=-1, vmax=1)
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(f"neg={neg_label}", rotation=0, labelpad=60, va="center", fontsize=8)

plt.suptitle("Negative conditioning: generate '3' while steering away from different classes")
plt.tight_layout(); plt.show()

## 4 · Mini ControlNet Simulation

Add an **edge map** (Sobel filter output of source image) as an extra input channel, training the model to match both the edge structure and the class label.

In [ ]:
import torch.nn.functional as F

def sobel_edges(x):
    """Simple edge detector on (B,1,H,W) images."""
    kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
    ky = kx.transpose(2, 3)
    gx = F.conv2d(x, kx, padding=1)
    gy = F.conv2d(x, ky, padding=1)
    return (gx.pow(2) + gy.pow(2)).sqrt().clamp(-1, 1)

# Visualise an edge map
for x0, y0 in loader:
    sample = x0[:8]
    break

edges = sobel_edges(sample)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    axes[0,i].imshow(sample[i,0].numpy(), cmap="gray", vmin=-1,vmax=1)
    axes[0,i].axis("off")
    axes[1,i].imshow(edges[i,0].numpy(), cmap="gray")
    axes[1,i].axis("off")
axes[0,0].set_ylabel("Original", rotation=0, labelpad=50)
axes[1,0].set_ylabel("Edge map", rotation=0, labelpad=50)
plt.suptitle("Sobel edge maps — the ControlNet control signal analogy")
plt.tight_layout(); plt.show()

In [ ]:
# Mini ControlNet: accept 2-channel input (noisy image + edge map)
class EdgeCondUNet(nn.Module):
    """Like CondUNet but first conv accepts 2 input channels (image + edge)."""
    def __init__(self, base_ch=32, time_dim=64):
        super().__init__()
        self.class_embed = nn.Embedding(NUM_CLASSES + 1, time_dim)
        self.time_embed  = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim*4), nn.SiLU(),
            nn.Linear(time_dim*4, time_dim))
        C = base_ch
        self.enc1 = ResBlock(2, C,   time_dim)   # ← 2 input channels
        self.enc2 = ResBlock(C, C*2, time_dim)
        self.enc3 = ResBlock(C*2, C*4, time_dim)
        self.bot  = ResBlock(C*4, C*4, time_dim)
        self.dec3 = ResBlock(C*8, C*2, time_dim)
        self.dec2 = ResBlock(C*4, C,   time_dim)
        self.dec1 = ResBlock(C*2, C,   time_dim)
        self.out  = nn.Conv2d(C, 1, 1)
        self.down = nn.MaxPool2d(2)
        self.up   = nn.Upsample(scale_factor=2, mode="nearest")
    def forward(self, x, edge, t, c):
        te  = self.time_embed(t) + self.class_embed(c)
        inp = torch.cat([x, edge], dim=1)   # (B, 2, H, W)
        s1  = self.enc1(inp, te); s2 = self.enc2(self.down(s1), te)
        s3  = self.enc3(self.down(s2), te); b = self.bot(s3, te)
        d3  = self.dec3(torch.cat([self.up(b), s3], 1), te)
        d2  = self.dec2(torch.cat([self.up(d3), s2], 1), te)
        d1  = self.dec1(torch.cat([d2, s1], 1), te)
        return self.out(d1)

ctrl_model = EdgeCondUNet()
ctrl_optim = torch.optim.Adam(ctrl_model.parameters(), lr=2e-4)

ctrl_model.train()
for epoch in range(20):
    for x0, c in loader:
        t    = torch.randint(0, T_STEPS, (len(x0),))
        eps  = torch.randn_like(x0)
        xt   = sqrt_ab[t].view(-1,1,1,1)*x0 + sqrt_1mab[t].view(-1,1,1,1)*eps
        edge = sobel_edges(x0).detach()
        mask = torch.rand(len(x0)) < 0.10
        c_in = c.clone(); c_in[mask] = NULL_CLASS
        loss = F.mse_loss(ctrl_model(xt, edge, t, c_in), eps)
        ctrl_optim.zero_grad(); loss.backward(); ctrl_optim.step()
    if (epoch+1) % 5 == 0: print(f"Epoch {epoch+1}/20  loss={loss.item():.4f}")
print("Edge-conditioned U-Net trained.")

In [ ]:
@torch.no_grad()
def ctrl_sample(ctrl_model, target_class, source_img, n_steps=40, w=3.0):
    ctrl_model.eval()
    N    = source_img.shape[0]
    cls  = torch.full((N,), target_class, dtype=torch.long)
    null = torch.full((N,), NULL_CLASS,   dtype=torch.long)
    edge = sobel_edges(source_img)
    x    = torch.randn_like(source_img)
    ts   = torch.linspace(T_STEPS-1, 0, n_steps+1).long()
    for i in range(n_steps):
        t_cur  = ts[i]; t_next = ts[i+1]
        tb     = torch.full((N,), t_cur, dtype=torch.long)
        eps_c  = ctrl_model(x, edge, tb, cls)
        eps_u  = ctrl_model(x, edge, tb, null)
        eps    = eps_u + w * (eps_c - eps_u)
        ab_c   = alpha_bar[t_cur]; ab_n = alpha_bar[t_next]
        x0_p   = ((x - sqrt_1mab[t_cur]*eps) / sqrt_ab[t_cur]).clamp(-1,1)
        x      = ab_n.sqrt()*x0_p + (1-ab_n).sqrt()*eps
    return x

# Source: '0', generate '6' using the '0' edge map as control
src = zeros[:4]
out = ctrl_sample(ctrl_model, target_class=6, source_img=src)

fig, axes = plt.subplots(3, 4, figsize=(8, 6))
for col in range(4):
    axes[0, col].imshow(src[col,0].numpy(),              cmap="gray", vmin=-1, vmax=1)
    axes[1, col].imshow(sobel_edges(src)[col,0].numpy(), cmap="gray")
    axes[2, col].imshow(out[col,0].numpy(),              cmap="gray", vmin=-1, vmax=1)
    for row in range(3): axes[row,col].axis("off")

axes[0,0].set_ylabel("Source '0'",  rotation=0, labelpad=55, va="center")
axes[1,0].set_ylabel("Edge map",    rotation=0, labelpad=55, va="center")
axes[2,0].set_ylabel("Output '6'",  rotation=0, labelpad=55, va="center")
plt.suptitle("Mini ControlNet: generate '6' guided by '0' edge structure")
plt.tight_layout(); plt.show()

## 5 · PixelSmith v5 — Real ControlNet (optional)

```python
# Uncomment to run (downloads SD 1.5 + ControlNet Canny, ~5 GB, ~5 min CPU)
# subprocess.run([sys.executable, "-m", "pip", "install",
#                 "diffusers", "transformers", "accelerate", "controlnet-aux", "-q"], check=True)
#
# from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
# from controlnet_aux import CannyDetector
# from PIL import Image
# import requests, io
#
# controlnet = ControlNetModel.from_pretrained(
#     "lllyasviel/sd-controlnet-canny", torch_dtype=torch.float32)
# pipe = StableDiffusionControlNetPipeline.from_pretrained(
#     "runwayml/stable-diffusion-v1-5",
#     controlnet=controlnet, torch_dtype=torch.float32)
# pipe = pipe.to("cpu")
#
# canny = CannyDetector()
# # Load any sketch image here
# sketch = Image.new("RGB", (512, 512), "white")  # blank placeholder
# canny_img = canny(sketch)
#
# result = pipe(
#     prompt="a photorealistic studio portrait, cinematic lighting",
#     image=canny_img,
#     num_inference_steps=20,
#     guidance_scale=7.5,
# ).images[0]
# result.save("pixelsmith_v5.png")
# plt.imshow(result); plt.axis("off"); plt.show()
```

**Next:** [TextToVideo.md](../TextToVideo/TextToVideo.md) — add time.